In [2]:
# Import necessary libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

import swcol as sw
parse_tech, colors, tech_order, tech_colors = sw.template.get() # Template

In [3]:
model_inputs_path = '../../model/inputs/'
model_outputs_path = '../../model/outputs/'

In [4]:

gen_build_costs=pd.read_csv(model_inputs_path+'gen_build_costs.csv')
gen_build_costs = gen_build_costs[gen_build_costs['build_year'] <= 2023]

gen_info=pd.read_csv(model_inputs_path+'gen_info.csv')
gen_build_costs = pd.merge(gen_build_costs, gen_info, on='GENERATION_PROJECT')

gen_build_costs['gen_tech'] = gen_build_costs['gen_tech'].replace(parse_tech)
gen_costs = gen_build_costs[['gen_tech','gen_overnight_cost','gen_fixed_om','gen_variable_om']]
print('Shape',gen_costs.shape)
gen_costs.sample(5)

Shape (241, 4)


,gen_tech,gen_overnight_cost,gen_fixed_om,gen_variable_om
169,Hydro,3083000,43780,1.39
201,Solar,1327000,15970,1.00
96,Hydro,3083000,43780,1.39
158,Hydro,3083000,43780,1.39
202,Solar,1327000,15970,1.00


In [5]:
# Calculate mean and standard error
gen_overnight_cost = gen_costs.groupby('gen_tech')['gen_overnight_cost'].agg(['mean', 'std', 'sem']).reset_index()
gen_overnight_cost = gen_overnight_cost.rename(columns={'mean': 'gen_overnight_cost', 'std': 'standard_dev','sem': 'error'})
# Apply desired order
gen_overnight_cost['gen_tech'] = pd.Categorical(gen_overnight_cost['gen_tech'], categories=tech_order, ordered=True)
gen_overnight_cost = gen_overnight_cost.sort_values('gen_tech')
# Plot
fig = go.Figure()
fig.add_trace(go.Bar(
    x=gen_overnight_cost['gen_tech'], y=gen_overnight_cost['gen_overnight_cost'],
    error_y=dict(type='data', array=gen_overnight_cost['error']),
    marker_color=colors[0]
))
fig.update_layout(
    height=12*50, width=16*50, template='plotly_white',
    title=dict(text='Average Overnight Cost', font=dict(size=24)),
    xaxis_title='Generation Technology', yaxis_title='Mean Overnight Cost [MUSD/MW]',
    xaxis=dict(title_font=dict(size=18), categoryorder='array', categoryarray=tech_order),
    yaxis=dict(title_font=dict(size=18))
)
fig.write_image("../images/Average Overnight Cost with Error Bars.png")
fig.show()

In [6]:
gen_overnight_cost['standard_dev'] = gen_overnight_cost['standard_dev'] / 1000000
gen_overnight_cost

,gen_tech,gen_overnight_cost,standard_dev,error
0,Hydro,2.626893e+06,0.773819,67608.908901
1,Run of River,2.092500e+06,0.903063,225765.660217
3,Thermal,1.886308e+06,0.804990,128901.609916
2,Solar,1.327000e+06,0.000000,0.000000
4,Wind,1.718000e+06,0.000000,0.000000


In [7]:
n_of_projects = gen_info.copy()
n_of_projects['n'] = 1
n_of_projects = n_of_projects.groupby(['gen_tech']).agg({'n': 'sum'}).reset_index()
n_of_projects

,gen_tech,n
0,Eolica,9
1,Hidro,131
2,RunOfRiver,16
3,Thermal,42
4,pv_solar,67


In [8]:
# Calculate mean and standard error
gen_fixed_om = gen_costs.groupby('gen_tech')['gen_fixed_om'].agg(['mean', 'std', 'sem']).reset_index()
gen_fixed_om = gen_fixed_om.rename(columns={'mean': 'gen_fixed_om', 'std': 'standard_dev', 'sem': 'error'})
# Apply desired order
gen_fixed_om['gen_tech'] = pd.Categorical(gen_fixed_om['gen_tech'], categories=tech_order, ordered=True)
gen_fixed_om = gen_fixed_om.sort_values('gen_tech')
# Plot
fig = go.Figure()
fig.add_trace(go.Bar(
    x=gen_fixed_om['gen_tech'], y=gen_fixed_om['gen_fixed_om'],
    error_y=dict(type='data', array=gen_fixed_om['error']),
    marker_color=colors[0]
))
fig.update_layout(
    height=12*50, width=16*50, template='plotly_white',
    title=dict(text='Average Fixed O&M Costs', font=dict(size=24)),
    xaxis_title='Generation Technology', yaxis_title='Average Fixed O&M Costs [USD/MW year]',
    xaxis=dict(title_font=dict(size=18), categoryorder='array', categoryarray=tech_order),
    yaxis=dict(title_font=dict(size=18)))
fig.write_image("../images/Average Fixed O&M Costs.png")
fig.show()

In [17]:
# Calculate mean and standard error
gen_variable_om = gen_costs.groupby('gen_tech')['gen_variable_om'].agg(['mean', 'std', 'sem']).reset_index()
gen_variable_om = gen_variable_om.rename(columns={'mean': 'gen_variable_om', 'std': 'standard_dev', 'sem': 'error'})
# Apply desired order
gen_variable_om['gen_tech'] = pd.Categorical(gen_variable_om['gen_tech'], categories=tech_order, ordered=True)
gen_variable_om = gen_variable_om.sort_values('gen_tech')
# Plot
fig = go.Figure()
fig.add_trace(go.Bar(
    x=gen_variable_om['gen_tech'], y=gen_variable_om['gen_variable_om'],
    error_y=dict(type='data', array=gen_variable_om['error']),
    marker_color=colors[0]
))
fig.update_layout(
    height=12*50, width=16*50, template='plotly_white',
    title=dict(text='Average Variable O&M Costs', font=dict(size=24)),
    xaxis_title='Generation Technology',
    yaxis_title='Average Variable O&M Costs [USD/MWh]',
    xaxis=dict(title_font=dict(size=18), categoryorder='array', categoryarray=tech_order),
    yaxis=dict(title_font=dict(size=18))
)
fig.write_image("../images/Average Variable O&M Costs.png")
fig.show()

In [18]:
gen_variable_om

,gen_tech,gen_variable_om,standard_dev,error
0,Hydro,1.379389,0.121445,0.010611
1,Run of River,1.390000,0.000000,0.000000
3,Thermal,8.070967,0.868161,0.139017
2,Solar,1.000000,0.000000,0.000000
4,Wind,0.000000,0.000000,0.000000


In [11]:
import pandas as pd
import plotly.graph_objects as go
# Merge DataFrames
df = gen_overnight_cost.merge(gen_fixed_om, on='gen_tech').merge(gen_variable_om, on='gen_tech')
df['gen_overnight_cost'] = df['gen_overnight_cost'] / 1000000
df['gen_fixed_om'] = df['gen_fixed_om'] / 1000
# Create the table
header = ['<b>Generation Technology</b>', '<b>Mean Overnight Cost [MUSD/MW]</b>', '<b>Mean Fixed O&M Costs [USD/MW year]</b>', '<b>Mean Variable O&M Costs [USD/MWh]</b>']
cells = [
    df['gen_tech'].tolist(),
    df['gen_overnight_cost'].round(2).tolist(),
    df['gen_fixed_om'].round(2).tolist(),
    df['gen_variable_om'].round(4).tolist()
]

fig = go.Figure(data=[go.Table(
    header=dict(values=header, fill_color='lightgray', align='left', line_color='black'),
    cells=dict(values=cells, fill_color='white', align='left', line_color='black')
)])

fig.update_layout(title='Costs by Technology', height=9*50, width=16*50, template="plotly_white")
fig.write_image("../images/Costs by Technology.png")
fig.show()


In [12]:
fuel_cost = pd.read_csv(model_inputs_path + 'fuel_cost.csv')
# Replace names
fuel_cost['fuel'] = fuel_cost['fuel'].replace({
    'ACPM': 'Diesel',
    'CARBON': 'Coal',
    'COMBUSTOLEO': 'Fuel Oil',
    'GASIMPOR': 'Imported Gas',
    'GASNACIO': 'National Gas'
})
# Calculate mean and error
fuel_stats = fuel_cost.groupby('fuel')['fuel_cost'].agg(['mean', 'sem']).reset_index()
fuel_stats = fuel_stats.rename(columns={'mean': 'fuel_cost', 'sem': 'error'})
fig = go.Figure()
fig.add_trace(go.Bar(
    x=fuel_stats['fuel'], y=fuel_stats['fuel_cost'],
    error_y=dict(type='data', array=fuel_stats['error']),
    marker_color=colors[0]
))
fig.update_layout(
    height=12*50, width=16*50, template='plotly_white', title=dict(text='Average Fuel Cost', font=dict(size=24)),
    xaxis_title='Fuel', yaxis_title='Average Fuel Cost [USD/MMBtu]',
    xaxis=dict(title_font=dict(size=18), categoryorder='array', categoryarray=tech_order),
    yaxis=dict(title_font=dict(size=18)))
fig.write_image("../images/Average Fuel Cost with Error Bars.png")
fig.show()

In [13]:
model_path = '../../../model/inputs/'
fuels = pd.read_csv(model_inputs_path+'fuels.csv')
fuels = fuels[['fuel','co2_intensity']]

# Lista de valores de interés
filtered_fuels = ['GASIMPOR', 'GASNACIO', 'ACPM', 'CARBON', 'COMBUSTOLEO', 'GLP']

# Filtra el DataFrame donde los valores de 'columna_deseada' están en la lista de interés
fuels = fuels[fuels['fuel'].isin(filtered_fuels)]
fuels['fuel'] = fuels['fuel'].str.replace('GASNACIO', 'National Gas')
fuels['fuel'] = fuels['fuel'].str.replace('GASIMPOR', 'Imported Gas')
fuels['fuel'] = fuels['fuel'].str.replace('ACPM', 'Diesel')
fuels['fuel'] = fuels['fuel'].str.replace('CARBON', 'Coal')
fuels['fuel'] = fuels['fuel'].str.replace('COMBUSTOLEO', 'Fuel Oil')
fuels['fuel'] = fuels['fuel'].str.replace('GLP', 'LPG')
import plotly.express as px
# Crear la gráfica de barras, usando 'gen_tech' como color
fig = px.bar(
    fuels, x='fuel', y='co2_intensity',
    color_discrete_sequence=[colors[0]],
    labels={'co2_intensity': 'CO2 Intensity [tCO2/MMBtu]', 'fuel': 'Fuel Type'},
    height=12*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')
fig.update_layout(
    title=dict(text='CO2 Intensity', font=dict(size=24)),
    xaxis=dict(title_font=dict(size=18), categoryorder='array', categoryarray=tech_order),
    yaxis=dict(title_font=dict(size=18)))
fig.write_image("../images/CO2 Intensity.png")
# Mostrar la gráfica
fig.show()

In [14]:
heat_rate = pd.read_csv(model_inputs_path+'gen_info.csv')
heat_rate = heat_rate[heat_rate['gen_tech'] == 'Thermal']
heat_rate = heat_rate[['gen_energy_source','gen_full_load_heat_rate']]
heat_rate['gen_full_load_heat_rate'] = pd.to_numeric(heat_rate['gen_full_load_heat_rate'], errors='coerce')
# Replace names
heat_rate['fuel'] = heat_rate['gen_energy_source'].replace({
    'ACPM': 'Diesel',
    'CARBON': 'Coal',
    'COMBUSTOLEO': 'Fuel Oil',
    'GASIMPOR': 'Imported Gas',
    'GASNACIO': 'National Gas'
})
heat_rate = heat_rate.groupby('fuel')['gen_full_load_heat_rate'].agg(['mean', 'sem']).reset_index()
heat_rate = heat_rate.rename(columns={'mean': 'heat_rate', 'sem': 'error'})
fig = go.Figure()
fig.add_trace(go.Bar(
    x=heat_rate['fuel'], y=heat_rate['heat_rate'],
    error_y=dict(type='data', array=heat_rate['error']),
    marker_color=colors[0]
))
fig.update_layout(
    height=12*50, width=16*50, template='plotly_white', title=dict(text='Average Heat Rate', font=dict(size=24)),
    xaxis_title='Fuel', yaxis_title='Average Heat Rate [MMBtu/MWh]',
    xaxis=dict(title_font=dict(size=18), categoryorder='array', categoryarray=tech_order),
    yaxis=dict(title_font=dict(size=18)))
fig.write_image("../images/Average Heat Rate.png")
fig.show()

heat_rate

,fuel,heat_rate,error
0,Coal,10.327400,0.756076
1,Diesel,5.359671,1.448968
2,Fuel Oil,12.332467,0.671448
3,Imported Gas,8.905860,0.970347
4,National Gas,8.308791,0.414415


In [15]:
# Calculate mean and error
fuel_stats = fuel_cost.groupby('fuel')['fuel_cost'].agg(['mean', 'sem']).reset_index()
fuel_stats = fuel_stats.rename(columns={'mean': 'fuel_cost', 'sem': 'error'})
fig = go.Figure()
fig.add_trace(go.Bar(
    x=fuel_stats['fuel'], y=fuel_stats['fuel_cost'],
    error_y=dict(type='data', array=fuel_stats['error']),
    marker_color=colors[0]
))
fig.update_layout(
    height=12*50, width=16*50, template='plotly_white', title=dict(text='Average Fuel Cost', font=dict(size=24)),
    xaxis_title='Fuel', yaxis_title='Average Fuel Cost [USD/MMBtu]',
    xaxis=dict(title_font=dict(size=18), categoryorder='array', categoryarray=tech_order),
    yaxis=dict(title_font=dict(size=18)))
fig.write_image("../images/Average Fuel Cost with Error Bars.png")
fig.show()

In [16]:
import pandas as pd
import plotly.graph_objects as go
# Merge DataFrames
df = fuels.merge(fuel_cost, on='fuel', how='outer')
df = df.fillna(0)
# Create the table
header = ['<b>Generation Technology</b>', '<b>Fuel Cost [USD/MMBtu]</b>', '<b>CO2 Intensity [tCO2/MMBtu year]</b>']
cells = [
    df['fuel'].tolist(),
    df['fuel_cost'].round(2).tolist(),
    df['co2_intensity'].round(2).tolist()
]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=header,
        fill_color='lightgray',
        align='left', line_color='black'
    ),
    cells=dict(
        values=cells,
        fill_color='white',
        align='left', line_color='black'
    )
)])

fig.update_layout(title='Fuels', height=9*50, width=16*50, template="plotly_white")
fig.write_image("../images/Fuels.png")
fig.show()